In [ ]:
import os

my_bucket = os.getenv('WORKSPACE_BUCKET')
my_bucket

In [ ]:
# Set the environment variable (IPython/Jupyter)
%env OWNEREMAIL=jacquelyn.willis@icahn.mssm.edu

import os
USER_NAME = os.getenv('OWNEREMAIL').split('@')[0].replace('.','-')

# Expose USER_NAME for %%bash
%env USER_NAME=$USER_NAME

In [ ]:
#cp plink files and mt to new workspace from old 
!gsutil -m cp -r "gs://fc-secure-6a913fb5-f07d-4a34-81ae-809e0a8db35e/data/"  "$WORKSPACE_BUCKET/plink_data"

In [ ]:
!gsutil ls {my_bucket}/plink_data

In [ ]:
with open("variants.txt", "w") as fh:
    fh.write("chr22:15528559:C:A\n")
    fh.write("chr22:15528559:C:G\n")

In [ ]:
#10 cases and 40 controls

%%bash
set -euo pipefail

gsutil cat "$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/merged.tsv" | \
awk 'BEGIN{
  FS=OFS="\t"; ncase=100; nctrl=300;
}
NR==1{
  for(i=1;i<=NF;i++){
    if($i=="FID") f=i;
    if($i=="IID") g=i;
    if($i=="meningitis") m=i;
  }
  if(!f || !g || !m){ print "[ERR] Need FID, IID, meningitis columns"; exit 1 }
  next
}
{
  if($m ~ /^1(\.0+)?$/ && c<ncase){ print $f,$g; c++; next }
  if($m ~ /^0(\.0+)?$/ && k<nctrl){ print $f,$g; k++ }
}
END{
  printf("[INFO] wrote %d cases and %d controls\n", c, k) > "/dev/stderr"
}' > keep_samples.txt

wc -l keep_samples.txt | awk '{print "[INFO] lines:",$1}'
head keep_samples.txt

gsutil cp keep_samples.txt "$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/keep_samples.txt"



In [ ]:
%%bash
set -euo pipefail

gsutil cat "$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/merged.tsv" | \
awk 'BEGIN{
  FS=OFS="\t"; ncase=200; nctrl=600;
}
NR==1{
  for(i=1;i<=NF;i++){
    if($i=="FID") f=i;
    if($i=="IID") g=i;
    if($i=="meningitis") m=i;
  }
  if(!f || !g || !m){ print "[ERR] Need FID, IID, meningitis columns"; exit 1 }
  next
}
{
  if($m ~ /^1(\.0+)?$/ && c<ncase){ print $f,$g; c++; next }
  if($m ~ /^0(\.0+)?$/ && k<nctrl){ print $f,$g; k++ }
}
END{
  printf("[INFO] wrote %d cases and %d controls\n", c, k) > "/dev/stderr"
}' > keep_samples.txt

wc -l keep_samples.txt | awk '{print "[INFO] lines:",$1}'
head keep_samples.txt

gsutil cp keep_samples.txt "$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/keep_samples.txt"


In [ ]:
%%bash
set -euo pipefail

NCASE=10
NCTRL=40
SRC="$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/merged.tsv"
OUT="keep_samples.txt"

gsutil cat "$SRC" | \
awk -v ncase="$NCASE" -v nctrl="$NCTRL" 'BEGIN{
  FS=OFS="\t"; IGNORECASE=1;
}
NR==1{
  for(i=1;i<=NF;i++){
    if($i=="FID") f=i;
    if($i=="IID") g=i;
    if($i=="meningitis") m=i;
  }
  if(!f || !g || !m){ print "[ERR] Need FID, IID, meningitis columns" > "/dev/stderr"; exit 1 }
  next
}
{
  # ensure exactly two columns out, unique per FID/IID
  key = $f SUBSEP $g
  if(seen[key]) next
  if($m ~ /^1(\.0+)?$/ && c < ncase){ print $f, $g; seen[key]=1; ++c; next }
  if($m ~ /^0(\.0+)?$/ && k < nctrl){ print $f, $g; seen[key]=1; ++k }
}
END{
  if(c < ncase || k < nctrl){
    printf("[ERR] only got %d/%d cases and %d/%d controls\n", c,ncase,k,nctrl) > "/dev/stderr"
    exit 2
  }
  printf("[INFO] wrote %d cases and %d controls\n", c, k) > "/dev/stderr"
}' > "$OUT"

wc -l "$OUT" | awk '{print "[INFO] lines:",$1}'
head "$OUT"

gsutil cp "$OUT" "$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/$OUT"


In [ ]:
# Run REGENIE on PLINK1 hard calls to test chr22:15528559 against a binary phenotype "meningitis".

%%writefile regenie_test.sh
#!/bin/bash

set -o errexit
set -o nounset

set -euo pipefail  # -e: exit on error; -u: error on unset vars; -o pipefail: fail if any piped cmd fails.

# -----------------------------------------------------------------------------
# REQUIRED INPUTS — replace the placeholders below:
#   <BED_PREFIX> : path prefix to your PLINK1 dataset (has .bed/.bim/.fam; NO extension)
#   <MERGED_TSV> : tab-delimited file with columns:
#                  FID  IID  meningitis  sex  age  PC1  PC2 ... PC10
#                  (Use this single file for both --phenoFile and --covarFile.)
# -----------------------------------------------------------------------------

!mkdir -p "/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/results"

# ================================
# STEP 1 — Fit LOCO predictors
# ================================

# Run REGENIE (binary trait pipeline).
regenie \
# Tell REGENIE we are doing Step 1 (fit ridge models + LOCO predictors).
  --step 1 \
# Use PLINK1 files: provide the *prefix* (no extension) that points to .bed/.bim/.fam.
  --bed $WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/plink/chr22.bed" \
# Binary trait mode: 0 = control, 1 = case; enables logistic models.
  --bt \
# Phenotype table path; we’ll select the single column named "meningitis".
  --phenoFile $WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/merged.tsv \
  --phenoCol meningitis \
# Covariate table path (can be the same file); we’ll specify which columns to use.
  --covarFile $WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/merged.tsv \
# Quantitative covariates to include (edit list if you have more/fewer PCs).
  --covarColList age,PC{1:10} \
# Categorical covariates to include (sex as M/F or 1/2 is fine).
  --catCovarList sex \
# Step-1 block size (1000 is a common default that balances speed/memory).
  --bsize 1000 \
# Reduce memory footprint by writing intermediates to disk (recommended).
  --lowmem \
# Number of CPU threads to use.
  --threads 8 \
# Prefix for Step-1 outputs (will create fit_step1.log, fit_step1_pred.list, etc.).
  --out $WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/menigitis_fit_step1

# ==============================================
# STEP 2 — Association test on the single SNP
# ==============================================

# Run REGENIE Step 2 using the predictors from Step 1.
regenie \
# Tell REGENIE we are doing Step 2 (single-variant association testing).
  --step 2 \
# Reuse the same PLINK1 dataset (hard calls) for testing.
  --bed chr22 \
# Binary trait mode again (logistic regression with Firth fallback).
  --bt \
# Use the same phenotype file/column as in Step 1.
  --phenoFile $WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/merged.tsv --phenoCol meningitis \
# Use the same covariate file and selections as in Step 1.
  --covarFile  $WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/merged.tsv \
  --covarColList age,PC{1:16} \
  --catCovarList sex \
# Feed in the LOCO predictor list produced by Step 1.
  --pred $WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/menigitis_fit_step1_pred.list \
# Restrict testing strictly to the SNP(s) listed here (our one-liner file).
  --extract $WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/variants.txt \
# Also restrict to chromosome 22 (helps avoid scanning other chromosomes).
  --chr 22 \
# Use Firth logistic regression with approximation for rare/imbalanced cases.
  --firth --approx \
# Step-2 block size (affects chunking through variants/samples).
  --bsize 2000 \
# Number of CPU threads to use.
  --threads 8 \
# Prefix for Step-2 outputs (creates assoc_rs17569141_meningitis.regenie[.gz]).
  --out assoc_chr22_15528559

# ============================
# OPTIONAL — quick inspection
# ============================

# Print header line of the summary output (handles gz or plain).
zcat -f assoc_chr22_15528559_meningitis.regenie.gz | head -n 1   # show column names
# Print the (likely single) result row (or all if your extract list had more).
zcat -f assoc_chr22_15528559_meningitis.regenie.gz | tail -n +2  # show results


In [ ]:
!regenie --version

In [ ]:
%%bash
dsub --version

In [ ]:
# Is the keep file empty?
!wc -l /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/keep_samples.txt

# Peek at the first few lines and show hidden characters (tabs, CRLF, commas)
!head -n 5 /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/keep_samples.txt | sed -n l

# See how IDs look in the .fam (FID IID are columns 1 & 2)
!head -n 5 /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/plink/chr22.fam


In [ ]:
!awk 'NR==FNR{a[$1" "$2]=1; next} (($1" "$2) in a){c++} END{print c}' \
  /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/keep_samples.txt \
  /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/plink/chr22.fam


In [ ]:
# from your current two-column keep file
!awk '{print 0, $2}' \
  /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/keep_samples.txt \
  > /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/keep_samples.fid0.txt


In [ ]:
!awk 'NR==FNR{a[$1" "$2]=1; next} ( ($1" "$2) in a ){c++} END{print c}' \
  /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/keep_samples.fid0.txt \
  /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/plink/chr22.fam


In [ ]:
import pandas as pd

pheno = pd.read_csv("merged.tsv", delimiter="\t")


In [ ]:
pheno["FID"] = "0"   
pheno

In [ ]:
pheno.to_csv("merged_wrangled.tsv", sep="\t", index=False)

In [ ]:
#clean1
import pandas as pd
import numpy as np
from pathlib import Path

in_path  = Path("/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/merged.tsv")
out_path = Path("/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/merged.regenie.tsv")

# 1) Read as strings so we control formatting; keep_default_na=False so "NA" stays literal if present
df = pd.read_csv(in_path, sep="\t", dtype=str, keep_default_na=False)

# 2) Basic header sanity
cols = [c.strip() for c in df.columns]
df.columns = cols
required = ["FID", "IID", "meningitis"]
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}. Found: {list(df.columns)}")

# 3) Strip whitespace in all cells
df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

# 4) Force FID="0"
df["FID"] = "0"

# 5) Turn empty strings into actual NA
df = df.replace(r"^\s*$", pd.NA, regex=True)

# 6) Ensure binary phenotype is 0/1 (Regenie --bt expects 0/1; NA allowed)
#    Convert common variants like "1.0" -> 1, "0.0" -> 0
def to01(s):
    if s is pd.NA or s is None:
        return pd.NA
    s = str(s).strip().lower()
    if s in {"1", "1.0", "case", "true", "yes"}:
        return 1
    if s in {"0", "0.0", "control", "false", "no"}:
        return 0
    # anything else becomes NA so Regenie can drop it
    return pd.NA

df["meningitis"] = df["meningitis"].map(to01).astype("Int64")

# 7) (Optional) coerce numeric covariates, if present; leave non-numeric as categorical
num_like = ["age"] + [f"PC{i}" for i in range(1,17) if f"PC{i}" in df.columns]
for c in num_like:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 8) Drop exact duplicate rows on IDs (just in case)
df = df.drop_duplicates(subset=["FID","IID"])

# 9) Put FID/IID first for clarity
first = ["FID","IID"]
rest = [c for c in df.columns if c not in first]
df = df[first + rest]

# 10) Write clean TSV with literal NA for missing values
df.to_csv(out_path, sep="\t", index=False, na_rep="NA")

print(f"Wrote {out_path}")
print(f"Rows: {len(df):,}  |  Columns: {len(df.columns)}")
print("Phenotype counts (excluding NA):")
print(df["meningitis"].value_counts(dropna=True))


In [ ]:
#clean2
import pandas as pd
import numpy as np
from pathlib import Path

in_path  = Path("/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/merged.regenie.tsv")
out_path = Path("/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/merged.regenie.cleaned.tsv")

# Read as strings so we control formatting
df = pd.read_csv(in_path, sep="\t", dtype=str, keep_default_na=False)

# Ensure required columns exist
for col in ["FID", "IID", "sex", "meningitis"]:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# 1) FID must be "0"
df["FID"] = "0"

# 2) Trim whitespace on all string cells
for c in df.columns:
    df[c] = df[c].astype(str).str.strip()

# 3) sex: set "PMI: SKIP" -> NA; keep only {M,F} else NA
df["sex"] = df["sex"].str.upper()
df.loc[df["sex"].isin(["PMI: SKIP", "PMI:SKIP"]), "sex"] = pd.NA
df.loc[~df["sex"].isin(["M", "F"]), "sex"] = pd.NA

# 4) meningitis: remove stray commas like "1 ," and coerce to {0,1,NA}
y = "meningitis"
df[y] = (df[y]
         .str.replace(",", "", regex=False)   # drop comma if present ("1 ,")
         .str.replace(r"\.0$", "", regex=True)
         .str.lower()
         .replace({"case":"1","control":"0","true":"1","false":"0","yes":"1","no":"0","na":""})
        )
df[y] = pd.to_numeric(df[y], errors="coerce").astype("Int64")
df[y] = df[y].where(df[y].isin([0,1]), pd.NA)

# (Optional) numeric covariates
for c in [col for col in ["age"] + [f"PC{i}" for i in range(1,17)] if col in df]:
    df[c] = pd.to_numeric(df[c].str.replace(",", "", regex=False), errors="coerce")

# Write clean TSV
df.to_csv(out_path, sep="\t", index=False, na_rep="NA")

print("Wrote:", out_path)
print("Bad 'sex' -> NA rows:", (df["sex"].isna()).sum())
print("Phenotype value counts (non-missing):")
print(df[y].value_counts(dropna=True))


In [ ]:
!which regenie

In [ ]:
!which conda


In [ ]:
%%writefile regenie_test.sh

#!/usr/bin/env bash
set -euo pipefail

# Local paths
PLINK_PREFIX="/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/plink/chr22"      # prefix only (has .bed/.bim/.fam)
MERGED_TSV="/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/merged.regenie.cleaned.tsv"        # columns: FID IID meningitis sex age
SAMPLES="/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/keep_samples.fid0.txt"
OUT_DIR="/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/results"

mkdir -p "$OUT_DIR"

# Build variants.txt from BIM, handle chr naming (22 vs chr22)
awk -v p=28725083 '($1==22 || $1=="chr22") && $4==p {print $2}' "${PLINK_PREFIX}.bim" > "${OUT_DIR}/variants.txt"
echo "[variants.txt]"; cat "${OUT_DIR}/variants.txt"

#building moc

# STEP 1 — LOCO predictors
regenie \
  --step 1 \
  --bed "${PLINK_PREFIX}" \
  --bt \
  --phenoFile "${MERGED_TSV}" --phenoCol meningitis \
  --covarFile  "${MERGED_TSV}" \
  --covarColList age \
  --catCovarList sex \
  --keep "${SAMPLES}" \
  --bsize 1000 \
  --lowmem \
  --threads 8 \
  --gz \
  --out "${OUT_DIR}/fit_step1"

# STEP 2 — single-variant test
regenie \
  --step 2 \
  --bed "${PLINK_PREFIX}" \
  --bt \
  --phenoFile "${MERGED_TSV}" --phenoCol meningitis \
  --covarFile  "${MERGED_TSV}" \
  --covarColList age \
  --catCovarList sex \
  --keep "${SAMPLES}" \
  --pred "${OUT_DIR}/fit_step1_pred.list" \
  --extract "${OUT_DIR}/variants.txt" \
  --firth --approx \
  --mac 1 \
  --bsize 2000 \
  --threads 8 \
  --gz \
  --out "${OUT_DIR}/assoc_chr22"

# Quick peek
#zcat -f "${OUT_DIR}/assoc_chr22_meningitis.regenie.gz" | head -n 1
#zcat -f "${OUT_DIR}/assoc_chr22_meningitis.regenie.gz" | tail -n +2


In [ ]:
!conda config --prepend pkgs_dirs $HOME/.conda/pkgs
!conda config --prepend envs_dirs $HOME/.conda/envs


In [ ]:
!conda config --show pkgs_dirs
!conda config --show envs_dirs


In [ ]:
%%bash
# If you already have conda, just do the 'create' line; otherwise see the conda note below.
conda create -n regenie4 -y -c conda-forge -c bioconda regenie
conda activate regenie4
regenie --version          # should print 4.1.x
which -a regenie           # ensure the conda one is first on PATH



In [ ]:
!plink2 \
  --bfile plink/chr22 \
  --maf 0.01 --mac 100 --geno 0.1 --hwe 1e-15 \
  --mind 0.1 \
  --write-snplist --write-samples --no-id-header \
  --out qc_pass


In [ ]:
%%writefile regenie_test.sh

#!/usr/bin/env bash
set -euo pipefail

# Use REGENIE from your conda env (no need to activate)
export PATH="$HOME/.conda/envs/regenie4/bin:$PATH"


# Show which regenie we will use (sanity check)
echo "Using regenie at: $(which regenie)"


# Local paths
PLINK_PREFIX="/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/plink/chr22"      # prefix only (has .bed/.bim/.fam)
MERGED_TSV="/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/merged.regenie.cleaned.tsv"        # columns: FID IID meningitis sex age
VARIANTS="/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/qc_pass.snplist"
OUT_DIR="/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/2025-10-06_Pilot_infectious_disease_PheWAS/results"

mkdir -p "$OUT_DIR"


#STEP 1 - building moc

regenie \
  --step 1 \
  --bed "${PLINK_PREFIX}" \
  --bt --loocv \
  --phenoFile "${MERGED_TSV}" \
  --phenoCol meningitis \
  --covarFile "${MERGED_TSV}" \
  --covarColList  age,PC{1:16} \
  --catCovarList sex \
  --extract "${VARIANTS}" \
  --lowmem \
  --bsize 1000 \
  --threads 8 \
  --gz \
  --print-pheno \
  --out "${OUT_DIR}/fit_step1"

# STEP 2 — single-variant test
regenie \
  --step 2 \
  --bed "${PLINK_PREFIX}" \
  --bt --loocv \
  --phenoFile "${MERGED_TSV}" --phenoCol meningitis \
  --covarFile  "${MERGED_TSV}" \
  --covarColList age \
  --catCovarList sex \
  --pred "${OUT_DIR}/fit_step1_pred.list" \
  --extract "${VARIANTS}" \
  --firth --approx \
  --bsize 2000 \
  --threads 8 \
  --out "${OUT_DIR}/assoc_chr22"

# Quick peek
#zcat -f "${OUT_DIR}/assoc_chr22_meningitis.regenie.gz" | head -n 1
#zcat -f "${OUT_DIR}/assoc_chr22_meningitis.regenie.gz" | tail -n +2


In [ ]:
%%bash

chmod +x regenie_test.sh
./regenie_test.sh


In [ ]:
!zcat results/assoc_chr22_meningitis.regenie.gz


In [ ]:
#copying all needed data to my workspace bucket into a data folder

In [ ]:
%%bash

gsutil -m mkdir "$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/data"

In [ ]:
%%bash

gsutil -m cp -r /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_vaccinated_cohort_data_table.pkl "$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/"

In [ ]:
%%bash

gsutil -m cp -r /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_cohort_data_table.pkl "$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/"


In [ ]:
%%bash

gsutil -m cp -r /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/controls.pkl "$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/"

In [ ]:
%%bash

gsutil -m cp -r /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/controls_vax_data.pkl "$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/"

In [ ]:
%%bash

gsutil -m cp -r /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/seasonal_vaccine_COVID_cohort_data_table.pkl "$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/"

In [ ]:
%%bash

gsutil -m cp -r /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/seasonal_vaccine_influenza_cohort_data_table.pkl "$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/"

In [ ]:
%%bash

gsutil -m cp -r /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/seasonal_vaccine_influenza_seasonal_bin_cohort_data_table.pkl "$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/"

In [ ]:
%%bash

gsutil -m cp -r /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/seasonal_vaccine_COVID_strain_bin_cohort_data_table.pkl "$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/"

In [ ]:
%%bash

gsutil -m cp -r /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_B1_B2_data_table.pkl "$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/"


In [ ]:
%%bash

gsutil -m cp -r /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_vaccinated_C1_C2_data_table.pkl "$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/"


In [ ]:
%%bash

gsutil -m cp -r /home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_vaccinated_D1_D2_data_table.pkl "$WORKSPACE_BUCKET/notebooks/2025-10-06_Pilot_infectious_disease_PheWAS/"


In [ ]:
!gsutil ls $WORKSPACE_BUCKET/data/

In [ ]:
!gsutil ls $WORKSPACE_BUCKET/data/phewas_hail_final.mt/